# OpenGeoSys 6 – Schulungsübung 1

**Ziel:** Stationäre Diffusion mit OGS rechnen, Ergebnis als VTU laden und mit PyVista, VTUinterface und Matplotlib auswerten. Optional: dieselbe Rechnung mit **PardisoLU** vergleichen.

Voraussetzung: `bash install.sh --all` oder die einzelnen Installationsphasen und `bash install.sh --smoke` wurden erfolgreich abgeschlossen. Als Kernel **Python (OGS Workshop)** wählen.

In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import pyvista as pv
import vtuIO
import ogstools as ot
from ogs6py import ogs

root = Path.home() / 'ogs-workshop'
results = root / 'results' / 'cases'
print('OGS tools:', getattr(ot, '__version__', 'loaded'))
print('Results:', results)
assert results.is_dir(), 'Zuerst bash install.sh --smoke ausführen'

## 1. Reale OGS-Resultate finden

Die Dateien sind beim automatischen numerischen Referenztest entstanden. Wir verwenden die **letzte VTU-Zeitstufe** des Diffusionsfalls.

In [ ]:
output = results / 'steady_diffusion'
vtus = sorted(output.glob('*.vtu'))
assert vtus, f'Keine VTU-Ausgabe in {output}'
file = vtus[-1]
mesh = pv.read(file)
print('Datei:', file.name)
print('Knoten:', mesh.n_points, 'Zellen:', mesh.n_cells)
print('Punktfelder:', list(mesh.point_data))
assert mesh.n_points > 0 and mesh.n_cells > 0

## 2. PyVista und VTUinterface

Wir wählen ein vorhandenes skalares Punktfeld und prüfen, ob beide Bibliotheken die gleichen Daten liefern.

In [ ]:
fields = [name for name in mesh.point_data if np.asarray(mesh[name]).ndim == 1]
assert fields, 'Kein skalares Punktfeld gefunden'
field = fields[0]
values_pv = np.asarray(mesh.point_data[field])
io = vtuIO.VTUIO(str(file), dim=2)
values_vtu = np.asarray(io.get_point_field(field))
print('Feld:', field, 'Shape:', values_pv.shape)
print('Maximale Abweichung:', float(np.max(np.abs(values_pv-values_vtu))))
assert np.allclose(values_pv, values_vtu, rtol=1e-12, atol=1e-12)

## 3. Ergebnisdiagramm mit Matplotlib

Dieser Plot ist bewusst einfach: eine Punktwolke der Feldwerte über der x-Koordinate. Für eine echte Schnittlinie kann später `sample_over_line` in PyVista verwendet werden.

In [ ]:
fig, ax = plt.subplots(figsize=(7,4))
ax.scatter(mesh.points[:,0], values_pv, s=20)
ax.set(xlabel='x [m]', ylabel=field, title='OGS: stationäre Diffusion')
ax.grid(alpha=0.25)
plot_path = root / 'results' / 'workshop_diffusion.png'
fig.savefig(plot_path, dpi=150, bbox_inches='tight')
plt.show()
print('Gespeichert:', plot_path)

## 4. Optionaler PARDISO-Vergleich

Wenn der zusätzliche OGS-PardisoLU-Test gelaufen ist, vergleichen wir die numerischen Ausgabefelder von SparseLU und PardisoLU. Dabei **berichten** wir die Abweichung, statt eine unbewiesene Exaktheitsforderung zu formulieren.

In [ ]:
pardiso_files = sorted((results/'pardiso_steady_diffusion').glob('*.vtu'))
if pardiso_files:
    pardiso = pv.read(pardiso_files[-1])
    vals2 = np.asarray(pardiso.point_data[field])
    assert vals2.shape == values_pv.shape
    print('SparseLU gegen PardisoLU: max |Δ| =', float(np.max(np.abs(vals2-values_pv))))
else:
    print('PardisoLU-Testdateien fehlen. Bitte bash install.sh --smoke ausführen.')

## 5. Was wurde wirklich getestet?

- OGS hat das Referenzmodell gerechnet und Ergebnisdateien geschrieben.
- PyVista und VTUinterface lesen dasselbe Feld konsistent ein.
- Matplotlib erzeugt ein Diagramm.
- `ogs6py` wurde importiert; die XML-Projektgenerierung wird separat durch `verify.py` getestet.

Die übrigen Prozessfälle (Small Deformation, Richards Flow, Hydro-Mechanics, TRM und TH2M) liegen in jeweils eigenen Ergebnisordnern und können analog untersucht werden.